# 04 — Cross-dataset evaluation, failure gallery, fairness

Prereqs: `02_train_dan.ipynb` and (ideally) `03_train_poster.ipynb` have produced `runs/dan_rafdb/best.pth` and `runs/poster_rafdb/best.pth`.

In [ ]:
%cd /content/fer
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from torch.utils.data import DataLoader
from src.data import CLASSES, FER2013Dataset, RAFDBDataset, build_transforms
from src.models import build_model
from src.eval import collect_predictions, confusion_matrix, per_class_metrics, plot_confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Cross-dataset evaluation
Train on RAF-DB, evaluate on FER-2013 PrivateTest (and vice versa). Both label spaces are aligned (see `scripts/prepare_rafdb.py`).

In [ ]:
def load_model(name, ckpt):
    m = build_model(name).to(device)
    state = torch.load(ckpt, map_location=device, weights_only=False)
    sd = state.get('model', state.get('model_state_dict', state))
    sd = {k.removeprefix('module.'): v for k, v in sd.items()}
    m.load_state_dict(sd, strict=False)
    return m.eval()

TRANSFORM = build_transforms(train=False, image_size=224)

def eval_on(model, ds_cls, root, split):
    ds = ds_cls(root, split=split, transform=TRANSFORM)
    loader = DataLoader(ds, batch_size=128, num_workers=2)
    pred, true = collect_predictions(model, loader, device)
    cm = confusion_matrix(pred, true, len(CLASSES))
    war = float((pred == true).mean())
    pcm = per_class_metrics(cm)
    uar = float(np.mean([m['recall'] for m in pcm]))
    return war, uar, cm

rows = []
for name, ckpt in [('dan', 'runs/dan_rafdb/best.pth'), ('poster_pp', 'runs/poster_rafdb/best.pth')]:
    if not Path(ckpt).exists():
        print(f'skip {name}: {ckpt} missing'); continue
    m = load_model(name, ckpt)
    for ds_name, ds_cls, root, split in [
        ('rafdb', RAFDBDataset, 'data/rafdb', 'test'),
        ('fer2013', FER2013Dataset, 'data/fer2013', 'test'),
    ]:
        if not Path(root).exists(): continue
        war, uar, _ = eval_on(m, ds_cls, root, split)
        rows.append({'model': name, 'eval_set': ds_name, 'WAR': war, 'UAR': uar})
        print(f'{name:>10s} on {ds_name:>8s}: WAR={war:.4f} UAR={uar:.4f}')

pd.DataFrame(rows)

## 2. Failure-mode gallery
Top-confident-wrong predictions per class on RAF-DB test.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image

ds = RAFDBDataset('data/rafdb', split='test', transform=TRANSFORM)
loader = DataLoader(ds, batch_size=128, num_workers=2)
model = load_model('dan', 'runs/dan_rafdb/best.pth')
wrong = []  # (confidence, true_label, pred_label, sample_idx)
with torch.no_grad():
    idx = 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        out = model(imgs)
        logits = out[0] if isinstance(out, (tuple, list)) else out
        probs = F.softmax(logits, dim=1).cpu()
        conf, pred = probs.max(dim=1)
        for i in range(len(labels)):
            if pred[i].item() != labels[i].item():
                wrong.append((conf[i].item(), labels[i].item(), pred[i].item(), idx + i))
        idx += len(labels)
wrong.sort(reverse=True)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (cf, t, p, i) in zip(axes.flat, wrong[:10]):
    rel, _ = ds.samples[i]
    img = Image.open(ds.root / rel)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'true={CLASSES[t]}\npred={CLASSES[p]} ({cf:.2f})', fontsize=9)
plt.tight_layout(); plt.show()

## 3. Fairness sanity check (RAF-DB age/gender/race attributes)
RAF-DB ships demographic attributes (report.md:2217). Disaggregate accuracy by group.
Requires `data/rafdb/Annotation/Patch/` (or equivalent) per-image attributes — adapt the loader to your actual download layout.

In [ ]:
# Adapt this cell to your RAF-DB download once you've inspected the EmoLabel/Annotation tree.
# Skeleton: build a per-sample attribute dict, then group `pred == true` by attribute.
print('TODO: parse the RAF-DB attribute files and compute per-group accuracy.')